# 事件标签数据审查
输入：用户提供的 event_labels_20261018.rar；本地 J-TEXT HDF5；既有源基模 TEST220 预测。
这里核对覆盖、时间和划分，并分析错误对应关系。A–F 为离线模型生成的候选事件，尚非人工物理真值。所有推导代码保留在同目录 audit.py；不修改训练模型。

In [1]:
from pathlib import Path
import json, csv
from decimal import Decimal
import pandas as pd
from IPython.display import display, Code
root = Path('/media/hetaisheng/044A81D94A81C83E/Fuison/outputs/event_labels_audit_20261019')
labels = pd.read_csv(root / 'input/event_labels_20261018/JTEXT_event_labels.csv', encoding='utf-8-sig')
events = labels[labels.annotation_status.eq('event')]
shots = labels.drop_duplicates('shot_no')
print({'rows':len(labels), 'shots':len(shots), 'events':len(events)})
display(pd.crosstab(shots.annotation_status, shots.disruption_label))

{'rows': 1975, 'shots': 1816, 'events': 732}


disruption_label,0,1
annotation_status,,
event,188,385
excluded_short_record,0,9
no_event_detected,1216,18


In [2]:
assert not labels.duplicated().any()
assert not events.duplicated(['shot_no','event_index']).any()
trace = json.loads((root / 'input/event_labels_20261018/work/event_provenance.json').read_text())
trace = {(r['shot_no'],r['event_index']):r for r in trace}
with (root/'input/event_labels_20261018/JTEXT_event_labels.csv').open(encoding='utf-8-sig', newline='') as f:
    for row in csv.DictReader(f):
        if row['annotation_status'] != 'event': continue
        r = trace[(int(row['shot_no']),int(row['event_index']))]
        assert row['event_class'] == r['family']
        assert Decimal(row['event_start_s']) == Decimal('0.15') + Decimal(r['start_token'])*Decimal('0.005')
        assert Decimal(row['event_end_s']) == Decimal('0.15') + Decimal(r['end_token'])*Decimal('0.005')
print('732 个区间的类别与 token 时间换算一致；这不是对物理正确性的验证。')

732 个区间的类别与 token 时间换算一致；这不是对物理正确性的验证。


In [3]:
link = pd.read_csv(root/'shot_linkage.csv')
display(pd.crosstab(link.canonical_role,link.cohort))
display(pd.crosstab(link.hdf5_found,link.csv_label))
display(pd.read_csv(root/'family_summary.csv'))
summary = json.loads((root/'summary.json').read_text())
print(json.dumps(summary['timing'], ensure_ascii=False, indent=2))

cohort,sanity,training_pool
canonical_role,,
outside_canonical,7,76
test,18,202
train,136,1217
val,21,139


csv_label,0,1
hdf5_found,,
False,49,34
True,1355,378


,family,events,shots,disruptive_shots,safe_shots,pool_events,sanity_events,duration_median_ms,start_lead_median_ms,end_lead_median_ms,shot_min,shot_max
0,A,40,31,17,14,35,5,85.0,241.294919,3.325000,1054592,1080564
1,B,27,24,17,7,24,3,160.0,236.959848,2.459798,1053034,1057273
2,C,218,202,105,97,195,23,55.0,215.359508,76.859358,1051571,1057983
3,D,317,291,236,55,287,30,140.0,163.561938,3.055488,1051501,1058073
4,E,85,76,64,12,77,8,80.0,50.354008,1.768368,1051763,1080602
5,F,45,42,29,13,37,8,75.0,86.873858,2.817513,1051684,1058057


{
  "duration_ms": {
    "n": 732,
    "min": 4.999999999999977,
    "q25": 39.99999999999998,
    "median": 104.99999999999999,
    "q75": 199.99999999999997,
    "max": 480.0
  },
  "first_disruptive_event_start_lead_ms": {
    "n": 351,
    "min": 15.373507683715925,
    "q25": 96.9158526837159,
    "median": 178.84815768371593,
    "q75": 255.0055426837159,
    "max": 503.03323768371587
  },
  "scope": "Lead times and label-window comparisons use only matched local disruptive HDF5 records.",
  "disruptive_event_starts_before_source_positive": 423,
  "disruptive_event_total": 467,
  "event_duration_fraction_outside_source_positive": 0.8198092611580522,
  "disruptive_event_duration_ms": 56495.0,
  "source_positive_overlap_ms": 10179.875790875836,
  "starts_at_165ms": 143,
  "disruptive_event_end_within_5ms_of_down": 338,
  "ends_within_5ms_of_record_end": 436,
  "first_event_lead_gt30_shots": 333,
  "first_event_lead_gt10_shots": 351
}


In [4]:
predictions = pd.read_csv(root/'source_test220_errors_joined.csv')
display(pd.crosstab(predictions.category,predictions.status))
display(predictions[predictions.category.isin(['FP','FN_or_late'])])
print('这些是既有开发结果的事后对照，不能解释为使用标签后的预测提升。')

status,event,excluded_short_record,no_event_detected
category,,,
FN_or_late,8,0,2
FP,9,0,4
TN,8,0,89
TP,97,2,1


,shot_no,is_disrupt,alarm,lead_ms,timely,category,cohort,status,families
12,1052127,True,True,5.773668,False,FN_or_late,training_pool,event,F
19,1052585,False,True,NaN,False,FP,training_pool,event,D
23,1052853,False,True,NaN,False,FP,training_pool,event,D
27,1052945,False,True,NaN,False,FP,training_pool,no_event_detected,NaN
36,1053004,True,False,NaN,False,FN_or_late,training_pool,no_event_detected,NaN
46,1053197,False,True,NaN,False,FP,training_pool,event,D
51,1053308,False,True,NaN,False,FP,training_pool,event,D
52,1053313,True,False,NaN,False,FN_or_late,training_pool,event,C
54,1053413,False,True,NaN,False,FP,training_pool,event,B
58,1053761,True,True,6.474498,False,FN_or_late,sanity,event,C


这些是既有开发结果的事后对照，不能解释为使用标签后的预测提升。


In [5]:
display(Code(filename=str(root/'audit.py'), language='python'))
print('从项目根目录复跑：/home/hetaisheng/miniconda3/envs/rl_env/bin/python outputs/event_labels_audit_20261019/audit.py')

"""Read-only audit of supplied offline event labels against local J-TEXT HDF5.

Writes derived tables and descriptive statistics only in this analysis folder.
Does not execute supplied code, train a predictor, or alter labels/checkpoints.
"""
from pathlib import Path
from collections import Counter
from decimal import Decimal
import csv
import hashlib
import json

import h5py
import numpy as np
import pandas as pd

HERE = Path(__file__).resolve().parent
ROOT = HERE.parents[1]
INPUT = HERE / 'input/event_labels_20261018'


def quantiles(values):
    a = np.asarray(values, float)
    a = a[np.isfinite(a)]
    if not len(a):
        return {'n': 0}
    return dict(n=int(len(a)), **{k: float(v) for k, v in zip(
        ['min', 'q25', 'median', 'q75', 'max'], np.quantile(a, [0, .25, .5, .75, 1]))})


def records(df):
    return json.loads(df.to_json(orient='records'))


def run():
    path = INPUT / 'JTEXT_event_labels.csv'
    df = pd.read_csv(path, encoding='utf-8-sig')
    shots = df.drop_duplicates('shot_no').copy()
    events = df.loc[df.annotation_status.eq('event')].copy()
    events['duration_ms'] = (events.event_end_s - events.event_start_s) * 1000
    checks = {}
    checks['duplicate_rows'] = int(df.duplicated().sum())
    checks['duplicate_event_keys'] = int(events.duplicated(['shot_no', 'event_index']).sum())
    checks['inconsistent_shot_labels'] = int((df.groupby('shot_no').disruption_label.nunique() > 1).sum())
    checks['inconsistent_shot_cohorts'] = int((df.groupby('shot_no').cohort.nunique() > 1).sum())
    checks['invalid_intervals'] = int((events.event_end_s <= events.event_start_s).sum())
    checks['invalid_classes'] = int((~events.event_class.isin(list('ABCDEF'))).sum())
    blank_fields = ['event_index', 'event_class', 'event_start_s', 'event_end_s']
    checks['populated_non_event_fields'] = int(df.loc[~df.annotation_status.eq('event'), blank_fields].notna().sum().sum())
    overlaps = []
    for sid, g in events.groupby('shot_no'):
        g = g.sort_values('event_index')
        if (np.diff(g.event_index.to_numpy()) != 1).any() or g.event_index.iloc[0] != 1:
            overlaps.append({'shot': int(sid), 'issue': 'event_index'})
        if len(g) > 1 and (g.event_start_s.to_numpy()[1:] < g.event_end_s.to_numpy()[:-1] - 1e-10).any():
            overlaps.append({'shot': int(sid), 'issue': 'overlap'})
    checks['ordering_or_overlap_errors'] = overlaps
    provenance = json.loads((INPUT / 'work/event_provenance.json').read_text())
    trace = {(int(r['shot_no']), int(r['event_index'])): r for r in provenance}
    timing_errors = []
    with path.open(encoding='utf-8-sig', newline='') as f:
        for r in csv.DictReader(f):
            if r['annotation_status'] != 'event':
                continue
            t = trace[(int(r['shot_no']), int(r['event_index']))]
            for field, token in [('event_start_s', 'start_token'), ('event_end_s', 'end_token')]:
                if Decimal(r[field]) != Decimal('0.15') + Decimal(t[token]) * Decimal('0.005'):
                    timing_errors.append([r['shot_no'], field])
            if r['event_class'] != t['family']:
                timing_errors.append([r['shot_no'], 'family'])
    checks['provenance_mismatches'] = timing_errors
    checks['provenance_unique_keys'] = len(trace)
    payload = json.loads((INPUT / 'work/labels_payload.json').read_text())
    p = pd.DataFrame(payload['rows'], columns=payload['header'])
    checks['csv_payload_equal'] = bool(df.fillna('<missing>').equals(p.fillna('<missing>')))
    # Compare serialized values as types inferred by pandas can differ on blanks.
    checks['csv_payload_values_equal'] = bool((df.fillna('<missing>').to_numpy() == p.fillna('<missing>').to_numpy()).all())

    roles = json.loads((ROOT / 'artifacts/canonical/jtext_source_bundle_p131_v1/index.json').read_text())['split_shot_ids']
    role_by_shot = {int(s): role for role, ids in roles.items() for s in ids}
    advanced = json.loads((ROOT / 'shot_list/J-TEX

从项目根目录复跑：/home/hetaisheng/miniconda3/envs/rl_env/bin/python outputs/event_labels_audit_20261019/audit.py
